In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# This is the magic that lets you run the model in <10GB VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Set a padding token if one is not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Llama expects padding on the LEFT
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",  # Automatically maps the model to your GPU
)

# Recommended for training
model.config.use_cache = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Model loaded successfully!")

Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully!


All purpose prompt used in the model

In [2]:
def promptify(text_to_analyze):
    return f"""
    You are an AI content moderator for a US radio station. Your goal is to identify words and phrases that would be considered 'indecent' or 'profane' under FCC broadcast standards, making them unsuitable for airplay.

    You must consider the context. Do not flag words that are merely references to mature themes (like drugs or weapons) unless they are used in a particularly graphic, gratuitous, or shocking manner. Differentiate between a mere mention and content designed to shock.

    **Text to Analyze:**
    "{text_to_analyze}"

    Return a single JSON object with one key: "explicit_phrases_found" containing a list of each word or phrase EXACTLY as it appears in the text.

    Provide only the raw JSON object as your final response.
    """

def create_training_prompt(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that only returns valid JSON."},
        {"role": "user", "content": f"{promptify(prompt)}"},
    ]

    # Format to return the attention mask
    formatted_prompt = tokenizer.apply_chat_template( messages, add_generation_prompt=True, tokenize=False)

    return formatted_prompt
   

Prepare the model for fine tuning with LoRA. Only run this once!


In [3]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=32, 
    lora_alpha=32, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Getting an error in training without this
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

model.print_trainable_parameters()

trainable params: 27,262,976 || all params: 8,057,524,224 || trainable%: 0.3384


Load WMSE dataset

In [4]:
import json

# Read the synth data set
with open('formatted_WMSE_data.jsonl', 'r', encoding='utf-8') as file:
    data_list = [json.loads(line) for line in file]

# Create train and val sets
train_size = int(0.8 * len(data_list))

train_data = data_list[:train_size]
val_data = data_list[train_size:]

print("Train data:", len(train_data))
print("Test data:", len(val_data))

Train data: 604
Test data: 151


Dataset and collate functions

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class CustomJsonlDataset(Dataset):
    def __init__(self, data_list, tokenizer):
        self.data = data_list
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        model_response = {'explicit_phrases_found': item['explicit_phrases']}
        
        prompt_messages = [
            {"role": "system", "content": "..."}, 
            {"role": "user", "content": promptify(item['sentence'])}
        ]
        
        assistant_message = [{"role": "assistant", "content": json.dumps(model_response)}]
        
        prompt_tokens = self.tokenizer.apply_chat_template(
            prompt_messages, 
            tokenize=True, 
            add_generation_prompt=True,
            add_special_tokens=True
        )

        if prompt_tokens[-1] == self.tokenizer.eos_token_id:
            prompt_tokens = prompt_tokens[:-1]
        
        assistant_tokens = self.tokenizer.apply_chat_template(
            assistant_message, 
            tokenize=True, 
            add_special_tokens=False
        )
        
        input_ids = prompt_tokens + assistant_tokens
        labels = ([-100] * len(prompt_tokens)) + assistant_tokens
        
        # Return PyTorch tensors directly from the dataset
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }
    
def collate_fn(batch, tokenizer):
    input_ids_tensors = [item['input_ids'] for item in batch]
    labels_tensors = [item['labels'] for item in batch]

    padded_input_ids = pad_sequence(
        input_ids_tensors, 
        batch_first=True, 
        padding_value=tokenizer.pad_token_id
    )
    
    padded_labels = pad_sequence(
        labels_tensors, 
        batch_first=True, 
        padding_value=-100 # -100 for label padding
    )
    
    attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()

    return {
        'input_ids': padded_input_ids,
        'labels': padded_labels,
        'attention_mask': attention_mask
    }

Training parameters and dataloaders

In [9]:
from torch.optim import AdamW
from functools import partial

# Training parameters
num_epochs = 3
learning_rate = 1e-5
batch_size = 2

train_dataset = CustomJsonlDataset(train_data, tokenizer)
val_dataset = CustomJsonlDataset(val_data, tokenizer)

# Custom data collator with correct padding
data_collator = partial(collate_fn, tokenizer=tokenizer)

# Load the train and val dataloaders
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    collate_fn=data_collator, 
    shuffle=True,
)

val_dataloader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    collate_fn=data_collator,
)

optimizer = AdamW(model.parameters(), lr=learning_rate)

In [10]:
import math
from torch.amp import GradScaler, autocast
from tqdm import tqdm

scaler = GradScaler()

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    total_train_loss = 0
    for batch in tqdm(train_dataloader, desc=f"(Epoch {epoch+1} / {num_epochs}) Training "):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        with autocast(dtype=torch.bfloat16, device_type=device):
            outputs = model(**batch)
            loss = outputs.loss
        
        scaler.scale(loss).backward() 
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Prevents catastrophic forgetting !
        scaler.step(optimizer)  
        scaler.update()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)

    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            total_val_loss += loss.item()


    avg_val_loss = total_val_loss / len(val_dataloader)
    perplexity = math.exp(avg_val_loss)

    print(f"Average Training Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Perplexity: {perplexity:.4f}")
    print()

(Epoch 1 / 3) Training : 0it [00:00, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
(Epoch 1 / 3) Training : 302it [03:35,  1.40it/s]


Average Training Loss: 0.8405
Average Validation Loss: 0.1992
Validation Perplexity: 1.2205



(Epoch 2 / 3) Training : 302it [03:28,  1.45it/s]


Average Training Loss: 0.1090
Average Validation Loss: 0.0115
Validation Perplexity: 1.0116



(Epoch 3 / 3) Training : 302it [03:37,  1.39it/s]


Average Training Loss: 0.0388
Average Validation Loss: 0.0106
Validation Perplexity: 1.0106



---------------

Test

In [16]:
def process_text_with_llama(text_to_process):
    print("Input:", text_to_process)
    print()

    # Format for the llm
    messages = [
        {"role": "system", "content": "You are a helpful assistant that only returns valid JSON."},
        {"role": "user", "content": f"{promptify(text_to_process)}"},
    ]

    # Format to return the attention mask
    formatted_prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    # Generate response
    print("Generating response...")
    outputs = model.generate(
        **inputs,
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
    )

    # Decode the output and print
    response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    try:
        formatted_response = json.loads(response_text)
        print("JSON formatted response:", formatted_response)

    except:
        print("Problem. Raw text:")
        print(response_text)

    print()

text_to_process = """
    think I fucked your girlfriend once
    Maybe twice, I don't remember
"""
process_text_with_llama(text_to_process)

Input: 
    think I fucked your girlfriend once
    Maybe twice, I don't remember


Generating response...
JSON formatted response: {'explicit_phrases_found': ['fucked']}

